# **1- Import Libraries**

In [ ]:
import os
import torch
import pandas as pd
from sentence_transformers import SentenceTransformer

# **2- Load and Prepare Hadith Dataset**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
hadiths_folder = "/content/drive/MyDrive/NLP/NLP_V2_Final/Hadith_embedding"
path = f"{hadiths_folder}/hadiths.csv"
hadith_df = pd.read_csv(path)
print("Columns:", hadith_df.columns)

Columns: Index(['id', 'hadith_id', 'source', 'chapter_no', 'hadith_no', 'chapter',
       'chain_indx', 'text_ar', 'text_en'],
      dtype='object')


In [ ]:
print(len(hadith_df))
hadith_df = hadith_df.dropna(subset=["text_ar"])
print(len(hadith_df))


34441
34433


In [ ]:
hadith_texts = hadith_df["text_ar"].astype(str).tolist()

# **3- Generate Hadith Embeddings using BGE-M3**

In [ ]:
# Load the BGE-M3 model
# Converts text into numerical embeddings (vectors)
embed_model = SentenceTransformer("BAAI/bge-m3")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [ ]:
# Encode all hadiths into embeddings
hadith_embeds = embed_model.encode(
    hadith_texts,   # A Python list of Arabic hadith texts
    convert_to_tensor=True, # Convert output to a PyTorch tensor instead of a Python list
    show_progress_bar=True, # Show progress bar while computing embeddings
    batch_size=64,  # Process 64 hadiths at a time for speed
    normalize_embeddings=True # Normalize vectors → required for cosine similarity
)

Batches:   0%|          | 0/539 [00:00<?, ?it/s]

# **4- Save Embeddings and Metadata**

In [ ]:
# Save the computed embeddings as a PyTorch tensor file
torch.save(hadith_embeds, f"{hadiths_folder}/hadith_embeds_bge.pt")

# Save the hadith metadata (text, source, id...) into a CSV file
hadith_df.to_csv(f"{hadiths_folder}/hadith_meta_bge.csv", index=False)

print("Hadith embeddings and metadata have been successfully saved.")

Hadith embeddings and metadata have been successfully saved.


# **5- Load Saved Embeddings for Use**

In [ ]:
hadiths_folder = "/content/drive/MyDrive/NLP/NLP_V2_Final/Hadith_embedding"

#Load the metadata CSV (hadith_meta_bge.csv)
hadith_df = pd.read_csv(f"{hadiths_folder}/hadith_meta_bge.csv")
hadith_texts = hadith_df["text_ar"].astype(str).tolist()

#Load the precomputed hadith embeddings
hadith_embeds = torch.load(f"{hadiths_folder}/hadith_embeds_bge.pt")

#Load the BGE-M3 embedding model
embed_model = SentenceTransformer("BAAI/bge-m3")